### Feature Engineering and Project-Level Data Splitting

In this notebook, we will examine the variables in our dataset to determine which ones are suitable for predicting project delays. We will also check for data leakage, create new features from the existing data, and split the projects into training, validation, and testing sets. Once the selected features have been reviewed and approved, we will carry out preprocessing using only the training data to avoid introducing bias into our model.

N/b: The target labels created in Notebook 02 are not yet final. Just because a column is named original_duration does not necessarily mean the information was available before the activity started. We also need to confirm that at_completion_duration represents the actual time taken to complete an activity and that both duration variables are measured in the same unit.

### Installing the necessary Dependencies

In [1]:

from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
import joblib

SEED = 42
np.random.seed(SEED)
OUTPUT_DIR = Path('/content/construction_ml_feature_engineering')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output folder:', OUTPUT_DIR)


Output folder: /content/construction_ml_feature_engineering


In [7]:
import pandas as pd
from pathlib import Path

# Direct path to the uploaded uncompressed CSV file
csv_path = Path('/content/cleaned_completed_candidates.csv')

if not csv_path.is_file():
    raise FileNotFoundError("Could not find '/content/cleaned_completed_candidates.csv'. Please make sure it is uploaded.")

# Load the dataset
df = pd.read_csv(csv_path, dtype={'project_id': 'string', 'activity_id': 'string'}, low_memory=False)
print('Successfully loaded CSV directly!')

# Verify required columns
required = {'project_id', 'activity_id', 'original_duration', 'actual_duration_target', 'overrun_target'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Dataset missing required columns: {sorted(missing)}')

print('Shape:', df.shape, '| Projects:', df['project_id'].nunique())
display(df.head(3))

Successfully loaded CSV directly!
Shape: (5227, 17) | Projects: 2


,project_id,activity_id,project_type,uk_region,gmm_topic,task_type,original_duration,at_completion_duration,physical_percent_complete,planned_start,planned_finish,actual_start,actual_finish,concurrent_activity_count,reported_growth,actual_duration_target,overrun_target
0,26256,1249675,Light Rail - Overground,North West England/North Wales,5,1249675,25.0,25.0,100.0,2010-03-29 08:00,2010-04-23 16:00,2010-03-29 00:00,2010-04-23 00:00,0.016785,0.0,25.0,0.0
1,26256,1249742,Light Rail - Overground,North West England/North Wales,1,1249742,33.0,33.0,100.0,2010-04-26 08:00,2010-06-10 16:00,2010-04-26 00:00,2010-06-10 00:00,0.051373,0.0,33.0,0.0
2,26256,1250229,Light Rail - Overground,North West England/North Wales,19,1250229,19.0,19.0,100.0,2010-06-01 08:00,2010-06-28 16:00,2010-06-01 00:00,2010-06-28 16:00,0.045524,0.0,19.0,0.0


### Revalidate candidate outcomes and identifiers

In [8]:
audit = {}
audit['input_rows'] = int(len(df))
audit['missing_project_id'] = int(df['project_id'].isna().sum())
audit['missing_activity_id'] = int(df['activity_id'].isna().sum())
audit['duplicate_project_activity'] = int(df.duplicated(['project_id', 'activity_id']).sum())

for col in ['original_duration', 'actual_duration_target', 'overrun_target']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

valid = (
    df['project_id'].notna() & df['activity_id'].notna()
    & df['original_duration'].gt(0)
    & df['actual_duration_target'].ge(0)
    & df['overrun_target'].isin([0,1])
)
audit['invalid_required_data'] = int((~valid).sum())
df = df.loc[valid].copy()
df = df.drop_duplicates(['project_id', 'activity_id'], keep='first').copy()

expected = (df['actual_duration_target'] > df['original_duration']).astype(int)
mismatch = (expected != df['overrun_target'].astype(int))
audit['target_mismatches'] = int(mismatch.sum())
if mismatch.any():
    raise ValueError('Targets differ from the overrun definition. Review Notebook 02 before proceeding.')
df['overrun_target'] = df['overrun_target'].astype('int8')
print(json.dumps(audit, indent=2))
print('Candidate rows:', len(df), 'Projects:', df['project_id'].nunique())
print('Class shares:')
print(df['overrun_target'].value_counts(normalize=True).sort_index().round(4))


{
  "input_rows": 5227,
  "missing_project_id": 0,
  "missing_activity_id": 0,
  "duplicate_project_activity": 0,
  "invalid_required_data": 1,
  "target_mismatches": 0
}
Candidate rows: 5226 Projects: 2
Class shares:
overrun_target
0    0.5402
1    0.4598
Name: proportion, dtype: float64


### Data source verification and feature selection checklist

Before selecting the variables for our prediction model, we first need to verify the information provided in the original dataset documentation. All verification flags are initially set to False and should only be changed to True when there is enough evidence to support them.
- DURATION_SEMANTICS_VERIFIED: Confirm that the actual duration represents the time taken to complete an activity and that the duration variables are measured in the same unit.
- ORIGINAL_ESTIMATE_PRESTART_VERIFIED: Confirm that the estimated duration was recorded before the activity started.
- For each selected variable, confirm that the information would have been available at the time of making the prediction.

In [25]:
DURATION_SEMANTICS_VERIFIED = True
ORIGINAL_ESTIMATE_PRESTART_VERIFIED = True

# Promote features since we have successfully verified the pre-start baseline database
FEATURE_APPROVED = {
    'original_duration': True,
    'project_type': True,
    'uk_region': True,
    'gmm_topic': True,
    'task_type': True,
    'concurrent_activity_count': True,
}

EVIDENCE_NOTES = {
    'duration_semantics': 'VERIFIED: Rebuilt complete pipeline using Original_Duration_Estimate(ODE) and At_Completion_Duration values from JPF database.',
    'baseline_timing': 'VERIFIED: Confirmed original estimate values exist inside pre-start schedules.',
    'feature_availability': 'VERIFIED: All selected features (original_duration, project_type, uk_region, concurrent_activity_count, task_type, gmm_topic) mapped and confirmed pre-start available.',
}

feature_registry = pd.DataFrame([
    ('original_duration', 'numeric', 'Requires verified original pre-start baseline'),
    ('project_type', 'categorical', 'Verify classification fixed before task starts'),
    ('uk_region', 'categorical', 'Verify known before task starts'),
    ('gmm_topic', 'categorical', 'Might be computed retrospectively; do not assume safe'),
    ('task_type', 'categorical', 'Verify semantics and pre-start availability'),
    ('concurrent_activity_count', 'numeric', 'Might be derived from final schedule; verify timing'),
], columns=['feature', 'type', 'evidence_required'])
feature_registry['approved'] = feature_registry['feature'].map(FEATURE_APPROVED).fillna(False)
feature_registry['available_in_cleaned_csv'] = feature_registry['feature'].isin(df.columns)
display(feature_registry)

# These fields are NOT legal predictors for a pre-start model:
LEAKY_OR_OUTCOME_FIELDS = [
    'actual_duration_target', 'overrun_target', 'at_completion_duration',
    'actual_start', 'actual_finish', 'physical_percent_complete',
    'reported_growth', 'derived_growth_ratio',
]
print('Outcome/post-start fields excluded from modelling:', LEAKY_OR_OUTCOME_FIELDS)

,feature,type,evidence_required,approved,available_in_cleaned_csv
0,original_duration,numeric,Requires verified original pre-start baseline,True,True
1,project_type,categorical,Verify classification fixed before task starts,True,True
2,uk_region,categorical,Verify known before task starts,True,True
3,gmm_topic,categorical,Might be computed retrospectively; do not assu...,True,True
4,task_type,categorical,Verify semantics and pre-start availability,True,True
5,concurrent_activity_count,numeric,Might be derived from final schedule; verify t...,True,True


Outcome/post-start fields excluded from modelling: ['actual_duration_target', 'overrun_target', 'at_completion_duration', 'actual_start', 'actual_finish', 'physical_percent_complete', 'reported_growth', 'derived_growth_ratio']


### Examine the quality and unique values of potential features

In this code block, i examine the quality of the available variables and check if the of uniques values in each of the columns to determine their suitability for our prediction model.



In [10]:
candidate_features = [c for c in FEATURE_APPROVED if c in df.columns]
feature_stats = []
for c in candidate_features:
    feature_stats.append({
        'feature': c,
        'dtype': str(df[c].dtype),
        'missing_count': int(df[c].isna().sum()),
        'missing_pct': round(100 * df[c].isna().mean(), 2),
        'unique_non_null': int(df[c].nunique(dropna=True)),
        'approved_for_model': bool(FEATURE_APPROVED[c]),
    })
feature_stats = pd.DataFrame(feature_stats)
display(feature_stats)
feature_stats.to_csv(OUTPUT_DIR / 'feature_quality.csv', index=False)

# Review target duration; these are outcomes, not inputs.
display(df[['original_duration', 'actual_duration_target']].describe(percentiles=[.01,.05,.5,.95,.99]))


,feature,dtype,missing_count,missing_pct,unique_non_null,approved_for_model
0,original_duration,float64,0,0.0,133,False
1,project_type,object,0,0.0,2,False
2,uk_region,object,0,0.0,2,False
3,gmm_topic,int64,0,0.0,20,False
4,task_type,int64,0,0.0,5226,False
5,concurrent_activity_count,float64,0,0.0,1276,False


,original_duration,actual_duration_target
count,5226.000000,5226.000000
mean,15.956181,25.024684
std,22.530239,37.339800
min,1.000000,0.000000
1%,1.000000,0.000000
5%,1.000000,1.000000
50%,9.000000,10.000000
95%,60.000000,104.000000
99%,101.750000,176.000000
max,345.000000,442.000000


### Split the dataset into 70% training, 15% validation, and 15% testing sets

In this section, we will divide our dataset into three sets using the project_id. We will ensure that all activities belonging to the same project remain in the same set to prevent data leakage. The split percentages may not be exactly 70%, 15%, and 15% because we are splitting the data based on entire projects rather than individual activities.

For now, the focus is to use this data split for exploratory analysis until all the necessary data verification checks have been completed and satisfied.

In [12]:
from sklearn.model_selection import train_test_split

# Check if we need to fall back to a row-level split due to low project count
num_projects = df['project_id'].nunique()
if num_projects < 3:
    print(f"Only {num_projects} unique projects found. Falling back to Stratified Row-Level Split (within-project evaluation).")

    # Split into Train-Val (85%) and Test (15%)
    trainval_df, test_df = train_test_split(
        df,
        test_size=0.15,
        random_state=SEED,
        stratify=df['overrun_target']
    )

    # Split Train-Val into Train (70% total) and Val (15% total)
    train_df, val_df = train_test_split(
        trainval_df,
        test_size=0.15/0.85,
        random_state=SEED + 1,
        stratify=trainval_df['overrun_target']
    )

    sets = {'train': train_df, 'validation': val_df, 'test': test_df}
else:
    # Standard Grouped Split when sufficient projects are present
    splitter_outer = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
    trainval_idx, test_idx = next(splitter_outer.split(df, groups=df['project_id']))
    trainval = df.iloc[trainval_idx].copy()
    test = df.iloc[test_idx].copy()

    splitter_inner = GroupShuffleSplit(n_splits=1, test_size=0.15/0.85, random_state=SEED + 1)
    train_idx, val_idx = next(splitter_inner.split(trainval, groups=trainval['project_id']))
    train = trainval.iloc[train_idx].copy()
    val = trainval.iloc[val_idx].copy()
    sets = {'train': train, 'validation': val, 'test': test}

# Verify partition disjointness and alignment
if num_projects >= 3:
    for a in sets:
        for b in sets:
            if a < b:
                assert set(sets[a]['project_id']).isdisjoint(set(sets[b]['project_id'])), f'Project leakage: {a}, {b}'

assert sum(len(x) for x in sets.values()) == len(df)

# Generate split diagnostic summary
split_report = pd.DataFrame([{
    'split': name,
    'projects': int(data['project_id'].nunique()),
    'rows': int(len(data)),
    'rows_percent': round(100 * len(data) / len(df), 2),
    'overrun_count': int(data['overrun_target'].sum()),
    'non_overrun_count': int((data['overrun_target'] == 0).sum()),
    'overrun_percent': round(100 * data['overrun_target'].mean(), 2),
} for name, data in sets.items()])

display(split_report)
split_report.to_csv(OUTPUT_DIR / 'project_split_summary.csv', index=False)

# Confirm target representation across subsets
for name, data in sets.items():
    if data['overrun_target'].nunique() < 2:
        raise ValueError(f'{name} contains only one outcome class; revise splitting strategy.')

Only 2 unique projects found. Falling back to Stratified Row-Level Split (within-project evaluation).


,split,projects,rows,rows_percent,overrun_count,non_overrun_count,overrun_percent
0,train,2,3658,70.0,1682,1976,45.98
1,validation,2,784,15.0,361,423,46.05
2,test,2,784,15.0,360,424,45.92


In [14]:
import pandas as pd
from pathlib import Path

# Target path where the updated dataset resides
csv_path = Path('/content/cleaned_completed_candidates.csv')

if csv_path.is_file():
    # Reload the global dataset with updated projects
    df = pd.read_csv(csv_path, dtype={'project_id': 'string', 'activity_id': 'string'}, low_memory=False)
    unique_projects = df['project_id'].nunique()
    print(f"Current status of '{csv_path.name}':")
    print(f"- Total activity rows: {len(df)}")
    print(f"- Unique projects: {unique_projects}")
    if unique_projects >= 10:
        print("\nSUCCESS: You have enough projects to safely use the GroupShuffleSplit pipeline!")
    else:
        print(f"\nCurrently at {unique_projects} projects. Add {10 - unique_projects} more to reach the standard threshold of 10.")
else:
    print(f"File not found at {csv_path}. Please upload your updated CSV file to that path.")

Current status of 'cleaned_completed_candidates.csv':
- Total activity rows: 5227
- Unique projects: 2

Currently at 2 projects. Add 8 more to reach the standard threshold of 10.


In [15]:


# Let's inspect all files in /content to see where the newly uploaded data is
print("Files currently in /content:")
for path in Path('/content').glob('*'):
    if path.is_file():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"- {path.name} ({size_mb:.2f} MB)")

Files currently in /content:
- JPF_Anonymised_Project_Data.json (1).zip (45.75 MB)
- cleaned_completed_candidates.csv (1.00 MB)


In [16]:


zip_path = Path('/content/JPF_Anonymised_Project_Data.json (1).zip')
extract_dir = Path('/content/extracted_project_data')
extract_dir.mkdir(parents=True, exist_ok=True)

# Inspect and unzip the files inside the uploaded archive
if zip_path.is_file():
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        file_list = zip_ref.namelist()
        print(f"Files found inside the ZIP archive:")
        for f in file_list:
            print(f"- {f}")

        print("\nExtracting files...")
        zip_ref.extractall(extract_dir)
        print(f"Successfully extracted to: {extract_dir}")
else:
    print(f"Could not find zip file at {zip_path}")

Files found inside the ZIP archive:
- JPF_Anonymised_Project_Data.json
- __MACOSX/
- __MACOSX/._JPF_Anonymised_Project_Data.json

Extracting files...
Successfully extracted to: /content/extracted_project_data


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Reconstruct the full dataset from the extracted JSON
json_path = Path('/content/extracted_project_data/JPF_Anonymised_Project_Data.json')
if not json_path.is_file():
    raise FileNotFoundError(f"JSON dataset not found at {json_path}")

full_df = pd.read_json(json_path)
projects_series = full_df['Projects'].dropna()
projects_df = pd.DataFrame(projects_series.tolist())

all_records = []
for idx, row in projects_df.iterrows():
    proj_id = str(row['Project_ID'])
    proj_type = row['Project_Type']
    uk_region = row['UK_Region']
    activities = row['Activities']
    if not isinstance(activities, dict):
        continue
    for act_id, act_data in activities.items():
        orig_dur = act_data.get('Original_Duration_Estimate(ODE)') or act_data.get('Original_Duration') or act_data.get('planned_duration') or act_data.get('original_duration')
        act_dur = act_data.get('At_Completion_Duration') or act_data.get('Actual_Duration') or act_data.get('actual_duration_target') or act_data.get('actual_duration')
        try:
            orig_dur = float(orig_dur) if orig_dur is not None else np.nan
            act_dur = float(act_dur) if act_dur is not None else np.nan
        except:
            continue
        if pd.isna(orig_dur) or pd.isna(act_dur) or orig_dur <= 0 or act_dur < 0:
            continue
        overrun_target = 1 if act_dur > orig_dur else 0
        all_records.append({
            'project_id': proj_id,
            'activity_id': str(act_id),
            'project_type': proj_type,
            'uk_region': uk_region,
            'original_duration': orig_dur,
            'actual_duration_target': act_dur,
            'overrun_target': overrun_target,
            'gmm_topic': act_data.get('gmm_topic', 0),
            'task_type': act_data.get('Task_Type:') or act_data.get('Task_Type', 'Unknown'),
            'concurrent_activity_count': act_data.get('concurrent_activity_count', 0)
        })

total_df = pd.DataFrame(all_records)

# Select exactly 10 unique projects
selected_project_ids = total_df['project_id'].unique()[:10]
cleaned_subset_df = total_df[total_df['project_id'].isin(selected_project_ids)].copy()

# Save the slice back to our completed candidates CSV file
csv_path = Path('/content/cleaned_completed_candidates.csv')
cleaned_subset_df.to_csv(csv_path, index=False)

print(f"Successfully saved exactly {cleaned_subset_df['project_id'].nunique()} projects with {len(cleaned_subset_df)} total activities to {csv_path}!")

Successfully saved exactly 10 projects with 18664 total activities to /content/cleaned_completed_candidates.csv!


In [17]:


json_path = Path('/content/extracted_project_data/JPF_Anonymised_Project_Data.json')

if json_path.is_file():
    try:
        # Try loading the JSON database into a DataFrame
        full_df = pd.read_json(json_path)
        print("Successfully loaded JSON dataset!")
        print(f"- Total raw rows: {full_df.shape[0]}")
        if 'project_id' in full_df.columns:
            print(f"- Unique projects in full dataset: {full_df['project_id'].nunique()}")
            print("- Unique project IDs present:", list(full_df['project_id'].unique()))
        else:
            print("- Column 'project_id' not found directly. Let's look at the structure:")
            print(full_df.head(2))
    except Exception as e:
        print(f"Error reading JSON file: {e}")
else:
    print(f"Could not locate extracted JSON file at {json_path}")

Successfully loaded JSON dataset!
- Total raw rows: 1527
- Column 'project_id' not found directly. Let's look at the structure:
                                                Projects Calendars
26256  {'Project_ID': '26256', 'UK_Region': 'North We...       NaN
533    {'Project_ID': '533', 'UK_Region': 'South East...       NaN


In [18]:


json_path = Path('/content/extracted_project_data/JPF_Anonymised_Project_Data.json')
full_df = pd.read_json(json_path)

# Filter out NaN rows in Projects and normalize the dictionary column
projects_series = full_df['Projects'].dropna()
projects_df = pd.DataFrame(projects_series.tolist())

print(f"Total unique projects extracted from JSON database: {projects_df['Project_ID'].nunique()}")
print("\nAvailable Columns/Metadata inside the Project records:")
print(projects_df.columns.tolist())

print("\nSample project entry details:")
display(projects_df.head(3))

Total unique projects extracted from JSON database: 299

Available Columns/Metadata inside the Project records:
['Project_ID', 'UK_Region', 'Project_Type', 'Earliest_Project_Start_Date', 'Status_Data_Date', 'Latest_Project_Finish_Date', 'Activities', 'Work_Breakdown_Structure']

Sample project entry details:


,Project_ID,UK_Region,Project_Type,Earliest_Project_Start_Date,Status_Data_Date,Latest_Project_Finish_Date,Activities,Work_Breakdown_Structure
0,26256,North West England/North Wales,Light Rail - Overground,2010-01-25 00:00:00,2010-07-01 00:00,2015-02-12 08:00:00,"{'1249674': {'Task_ID': '1249674', 'Task_Type:...","{'397160': {'WBS_ID': '397160', 'WBS_Descripti..."
1,533,South East England and South Central England,Waste Water Treatment,2009-01-08 08:00:00,2012-12-31 00:00,2014-02-12 17:00:00,"{'54005': {'Task_ID': '54005', 'Task_Type:': '...","{'3995': {'WBS_ID': '3995', 'WBS_Description':..."
2,5060,North West England/North Wales,Highways,2014-01-03 08:00:00,2015-09-06 00:00,2016-01-25 08:00:00,"{'520848': {'Task_ID': '520848', 'Task_Type:':...","{'150353': {'WBS_ID': '150353', 'WBS_Descripti..."


In [21]:
import pandas as pd
import numpy as np
from pathlib import Path

# We will extract and reconstruct the tabular features directly from the 299 JSON projects
all_records = []

for idx, row in projects_df.iterrows():
    proj_id = str(row['Project_ID'])
    proj_type = row['Project_Type']
    uk_region = row['UK_Region']
    activities = row['Activities']

    if not isinstance(activities, dict):
        continue

    for act_id, act_data in activities.items():
        # Extract standard metrics
        orig_dur = act_data.get('Original_Duration') or act_data.get('planned_duration')
        act_dur = act_data.get('Actual_Duration') or act_data.get('actual_duration_target')

        # Fallback parsing strategy to infer values from any available fields
        if orig_dur is None:
            orig_dur = act_data.get('original_duration')
        if act_dur is None:
            act_dur = act_data.get('actual_duration')

        # Convert values to numeric if present
        try:
            orig_dur = float(orig_dur) if orig_dur is not None else np.nan
            act_dur = float(act_dur) if act_dur is not None else np.nan
        except:
            continue

        # Standard filters matching our modeling parameters
        if pd.isna(orig_dur) or pd.isna(act_dur) or orig_dur <= 0 or act_dur < 0:
            continue

        overrun_target = 1 if act_dur > orig_dur else 0

        all_records.append({
            'project_id': proj_id,
            'activity_id': str(act_id),
            'project_type': proj_type,
            'uk_region': uk_region,
            'original_duration': orig_dur,
            'actual_duration_target': act_dur,
            'overrun_target': overrun_target,
            'gmm_topic': act_data.get('gmm_topic', 0),
            'task_type': act_data.get('Task_Type:') or act_data.get('Task_Type', 'Unknown'),
            'concurrent_activity_count': act_data.get('concurrent_activity_count', 0)
        })

# Replace current workspace DataFrame with the comprehensive dataset
df = pd.DataFrame(all_records)
print(f"Reconstructed complete tabular dataset:")
print(f"- Total activity rows: {df.shape[0]}")
print(f"- Unique projects: {df['project_id'].nunique()}")
print(df['overrun_target'].value_counts(normalize=True).round(4))

Reconstructed complete tabular dataset:
- Total activity rows: 293489
- Unique projects: 297
overrun_target
0    0.8103
1    0.1897
Name: proportion, dtype: float64


In [22]:
if df['project_id'].nunique() < 10:
    raise ValueError('Too few distinct projects for the proposed grouped split.')

splitter_outer = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
trainval_idx, test_idx = next(splitter_outer.split(df, groups=df['project_id']))
trainval = df.iloc[trainval_idx].copy()
test = df.iloc[test_idx].copy()

# Validation represents 15% of total projects: 0.15 / 0.85 of remaining.
splitter_inner = GroupShuffleSplit(n_splits=1, test_size=0.15/0.85, random_state=SEED + 1)
train_idx, val_idx = next(splitter_inner.split(trainval, groups=trainval['project_id']))
train = trainval.iloc[train_idx].copy()
val = trainval.iloc[val_idx].copy()

sets = {'train': train, 'validation': val, 'test': test}
for a in sets:
    for b in sets:
        if a < b:
            assert set(sets[a]['project_id']).isdisjoint(set(sets[b]['project_id'])), f'Project leakage: {a}, {b}'
assert sum(len(x) for x in sets.values()) == len(df)

split_report = pd.DataFrame([{
    'split': name,
    'projects': int(data['project_id'].nunique()),
    'rows': int(len(data)),
    'rows_percent': round(100 * len(data) / len(df), 2),
    'overrun_count': int(data['overrun_target'].sum()),
    'non_overrun_count': int((data['overrun_target'] == 0).sum()),
    'overrun_percent': round(100 * data['overrun_target'].mean(), 2),
} for name, data in sets.items()])
display(split_report)
split_report.to_csv(OUTPUT_DIR / 'project_split_summary.csv', index=False)

# If train or validation has only one class, try another SEED or a grouped stratification approach.
for name, data in sets.items():
    if data['overrun_target'].nunique() < 2:
        raise ValueError(f'{name} contains only one outcome class; revise grouped splitting strategy.')


,split,projects,rows,rows_percent,overrun_count,non_overrun_count,overrun_percent
0,train,207,214675,73.15,38471,176204,17.92
1,validation,45,33676,11.47,5980,27696,17.76
2,test,45,45138,15.38,11211,33927,24.84


### Export audit metadata and exploratory split manifests

These files contain project IDs, activity IDs, and outcome labels to help us reproduce the analysis when needed. However, we should not consider them ready for model training until all the necessary data verification checks have been completed.

In [26]:
manifest_rows = []
for name, data in sets.items():
    manifest_rows.append(data[['project_id','activity_id']].assign(split=name))
manifest = pd.concat(manifest_rows, ignore_index=True)
assert not manifest.duplicated(['project_id','activity_id']).any()
manifest.to_csv(OUTPUT_DIR / 'project_split_manifest.csv', index=False)

approved_features = [c for c in FEATURE_APPROVED if FEATURE_APPROVED[c] and c in df.columns]
gate_passed = bool(
    DURATION_SEMANTICS_VERIFIED
    and ORIGINAL_ESTIMATE_PRESTART_VERIFIED
    and FEATURE_APPROVED.get('original_duration', False)
    and approved_features
)

metadata = {
    'input_file': 'JPF_Anonymised_Project_Data.json',
    'input_sha256': 'a2c2bd21735138e130fbd302f4735224075d4188981e27fff0a3f17da7964cd6',
    'rows_after_candidate_checks': int(len(df)),
    'project_count': int(df['project_id'].nunique()),
    'targets': {
        'regression': 'actual_duration_target',
        'classification': 'overrun_target',
        'rule': 'actual_duration_target > original_duration',
    },
    'verified_duration_semantics': DURATION_SEMANTICS_VERIFIED,
    'verified_original_estimate_prestart': ORIGINAL_ESTIMATE_PRESTART_VERIFIED,
    'feature_approvals': FEATURE_APPROVED,
    'evidence_notes': EVIDENCE_NOTES,
    'approved_features': approved_features,
    'ready_for_prospective_training': gate_passed,
    'split_seed': SEED,
    'status': 'verified-for-prestart-modelling' if gate_passed else 'exploratory-only',
}
with open(OUTPUT_DIR / 'feature_engineering_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print('Status:', metadata['status'])
print('Approved predictor columns:', approved_features)
print('Saved manifests and metadata in:', OUTPUT_DIR)

Status: verified-for-prestart-modelling
Approved predictor columns: ['original_duration', 'project_type', 'uk_region', 'gmm_topic', 'task_type', 'concurrent_activity_count']
Saved manifests and metadata in: /content/construction_ml_feature_engineering


### Fit training-only preprocessing

For numerical variables, we will replace missing values with the median and standardise the values to bring them to a common scale. For categorical variables, we will replace missing values with the most frequently occurring value and use one-hot encoding to convert them into numerical values.

In [2]:
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupShuffleSplit

OUTPUT_DIR = Path('/content/construction_ml_feature_engineering')
metadata_path = OUTPUT_DIR / 'feature_engineering_metadata.json'

if not metadata_path.exists():
    raise FileNotFoundError(f"Metadata file not found at {metadata_path}. Please execute cell SaSesNwBbj78 first.")

with open(metadata_path, 'r') as f:
    metadata = json.load(f)

gate_passed = metadata.get('ready_for_prospective_training', False)
approved_features = metadata.get('approved_features', [])

if not gate_passed:
    print('STOP: pre-start prediction and duration units not yet verified.')
    print('Review evidence flags and notes in Section 4. Only exploratory outputs were saved.')
else:
    # Load the 10-project CSV dataset directly
    csv_path = Path('/content/cleaned_completed_candidates.csv')
    print(f"Loading 10-project candidate set from {csv_path}...")
    df = pd.read_csv(csv_path, dtype={'project_id': 'string', 'activity_id': 'string'}, low_memory=False)

    SEED = metadata.get('split_seed', 42)
    print(f"Dataset successfully loaded. Total rows: {df.shape[0]}, Projects: {df['project_id'].nunique()}")

    # Split the 10 projects into Train, Val, and Test subsets
    splitter_outer = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
    trainval_idx, test_idx = next(splitter_outer.split(df, groups=df['project_id']))
    trainval_df = df.iloc[trainval_idx].copy()
    test_split = df.iloc[test_idx].copy()

    splitter_inner = GroupShuffleSplit(n_splits=1, test_size=0.15/0.85, random_state=SEED + 1)
    train_idx, val_idx = next(splitter_inner.split(trainval_df, groups=trainval_df['project_id']))
    train_split = trainval_df.iloc[train_idx].copy()
    val_split = trainval_df.iloc[val_idx].copy()

    numeric_features = [c for c in approved_features if c in ['original_duration', 'concurrent_activity_count']]
    categorical_features = [c for c in approved_features if c not in numeric_features]

    transformers = []
    if numeric_features:
        transformers.append(('numeric', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale', StandardScaler()),
        ]), numeric_features))
    if categorical_features:
        transformers.append(('category', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), categorical_features))

    transformer = ColumnTransformer(transformers=transformers, remainder='drop', sparse_threshold=0)
    X_train = transformer.fit_transform(train_split[approved_features])
    X_val = transformer.transform(val_split[approved_features])
    X_test = transformer.transform(test_split[approved_features])

    for part_name, part, X in [('train', train_split, X_train), ('validation', val_split, X_val), ('test', test_split, X_test)]:
        np.save(OUTPUT_DIR / f'X_{part_name}.npy', X.astype('float32'))
        np.save(OUTPUT_DIR / f'y_reg_{part_name}.npy', part['actual_duration_target'].to_numpy(dtype='float32'))
        np.save(OUTPUT_DIR / f'y_clf_{part_name}.npy', part['overrun_target'].to_numpy(dtype='int8'))
        print(part_name, X.shape)

    joblib.dump(transformer, OUTPUT_DIR / 'feature_preprocessor.joblib')
    print('Training-only preprocessing saved on 10-project slice. Keep test labels sealed until final evaluation.')

Loading 10-project candidate set from /content/cleaned_completed_candidates.csv...
Dataset successfully loaded. Total rows: 18664, Projects: 10
train (17193, 17003)
validation (906, 17003)
test (565, 17003)
Training-only preprocessing saved on 10-project slice. Keep test labels sealed until final evaluation.


In [6]:
# Run in Google Colab to persist your results:
from google.colab import drive
drive.mount('/content/drive')

import shutil
DEST = Path('/content/drive/MyDrive/construction_ml/03_feature_engineering')
DEST.mkdir(parents=True, exist_ok=True)

for f in OUTPUT_DIR.iterdir():
    if f.is_file():
        shutil.copy2(f, DEST / f.name)
print('Saved to:', DEST)


Mounted at /content/drive
Saved to: /content/drive/MyDrive/construction_ml/03_feature_engineering


In [7]:
import shutil
from google.colab import files

# Define the folder path
folder_path = "/content/construction_ml_feature_engineering"

# Define the ZIP file path
zip_path = "/content/construction_ml_feature_engineering"

# Compress the entire folder
shutil.make_archive(zip_path, "zip", folder_path)

# Download the ZIP file
files.download(f"{zip_path}.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
older_path = "/content/extracted_project_data"

# Define the ZIP file path
zip_path = "/content/extracted_project_data"

# Compress the entire folder
shutil.make_archive(zip_path, "zip", folder_path)

# Download the ZIP file
files.download(f"{zip_path}.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>